# Ridge regression: intuition and code

In [1]:
import warnings
import numpy as np
from sklearn.datasets import load_diabetes, make_regression
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import train_test_split
warnings.simplefilter("ignore")

## One input: alpha shrinks the slope

In [2]:
X, y = make_regression(n_samples=100, n_features=1, n_informative=1, noise=20, random_state=13)
for alpha in [0, 10, 100]:
    model = LinearRegression() if alpha == 0 else Ridge(alpha=alpha)
    model.fit(X, y)
    print(f"alpha {alpha:4d}: slope {model.coef_[0]:.2f}, intercept {model.intercept_:.2f}")

alpha    0: slope 27.83, intercept -2.29
alpha   10: slope 24.95, intercept -2.13
alpha  100: slope 12.93, intercept -1.42


## Diabetes data: coefficients on one split, and mean test R2 over 200 splits

In [3]:
X, y = load_diabetes(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=45)
for alpha in [0, 0.01, 100000]:
    model = LinearRegression() if alpha == 0 else Ridge(alpha=alpha)
    model.fit(X_train, y_train)
    print(f"one split, alpha {alpha:>8}: s1 {model.coef_[4]:8.1f}, s2 {model.coef_[5]:7.1f}, largest |coef| {np.abs(model.coef_).max():7.1f}")

# Ridge shrinks the coefficients as a whole: their overall size (Euclidean norm) falls as alpha grows,
# while single coefficients (age, s6) can grow for a while.
for alpha in [0.0001, 0.01, 1, 100]:
    c = Ridge(alpha=alpha).fit(X_train, y_train).coef_
    print(f"alpha {alpha:>7}: norm {np.linalg.norm(c):7.1f}, age {c[0]:5.1f}, s6 {c[9]:6.1f}")

# Mean test R2 over 200 random splits: small (40) vs full (353) training set.
alphas = [0, 0.01, 0.04, 0.1, 1, 100000]
for n in (40, 353):
    r2 = np.zeros(len(alphas))
    for s in range(200):
        Xa, Xb, ya, yb = train_test_split(X, y, train_size=n, random_state=s)
        for i, a in enumerate(alphas):
            m = LinearRegression() if a == 0 else Ridge(alpha=a)
            r2[i] += m.fit(Xa, ya).score(Xb, yb) / 200
    print(n, "training observations:", {a: round(v, 3) for a, v in zip(alphas, r2)})

one split, alpha        0: s1   -970.8, s2   573.5, largest |coef|   970.8
one split, alpha     0.01: s1   -421.0, s2   137.9, largest |coef|   581.5
one split, alpha   100000: s1      0.0, s2     0.0, largest |coef|     0.0
alpha  0.0001: norm  1540.4, age  23.5, s6   89.4
alpha    0.01: norm   997.8, age  26.3, s6   94.3
alpha       1: norm   482.1, age  43.0, s6  120.9
alpha     100: norm    15.4, age   2.5, s6    5.1


40 training observations: {0: np.float64(0.293), 0.01: np.float64(0.396), 0.04: np.float64(0.417), 0.1: np.float64(0.39), 1: np.float64(0.146), 100000: np.float64(-0.028)}


353 training observations: {0: np.float64(0.478), 0.01: np.float64(0.479), 0.04: np.float64(0.479), 0.1: np.float64(0.479), 1: np.float64(0.414), 100000: np.float64(-0.013)}
